# V8 — Structured Output in the NLQ Agent

**Goal:** Understand how our E-commerce NLQ agent returns a predictable contract to the supervisor.

> Pydantic fundamentals are covered in an earlier notebook. This notebook focuses on how the *real project schemas* are used in the agent workflow.

## What we will follow

1. `ModelDecision` — the model's structured judgment.
2. `ToolStrategy` — how LangChain asks the agent for that structure.
3. `structured_response` — where the parsed decision appears.
4. `NLQResponse` — how Python combines the decision with actual SQL results captured by tools.
5. Why the model's judgment and database facts are kept separate.


## 1. The two contracts have different jobs

Our implementation deliberately uses **two Pydantic models**:

| Model | Created by | Purpose |
|---|---|---|
| `ModelDecision` | Structured output from the agent/model | Judgments: answerable?, entities, note, unanswered parts |
| `NLQResponse` | Application code in `assemble()` | Final tabular payload: columns, rows, executed SQL, result status, and metadata |

**Important:** `ModelDecision` has no `rows` or `sql` field. The model cannot supply database rows as part of its decision. The rows and executed SQL come from the tool/state path.


In [ ]:
from typing import Any, Literal
from pydantic import BaseModel, Field

# Project schema: the model's judgment only.
class ModelDecision(BaseModel):
    entities: list[str] = Field(default_factory=list)
    answerable: bool = True
    note: str = ""
    unmet_parts: list[str] = Field(default_factory=list)

# Project schema: the final result delivered to the supervisor.
class TokenUsage(BaseModel):
    input_tokens: int = 0
    cached_input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    llm_calls: int = 0
    model_id: str = ""

class NLQResponse(BaseModel):
    result_shape: Literal["table", "empty", "unanswerable", "not_executed"]
    columns: list[str] = Field(default_factory=list)
    rows: list[list[Any]] = Field(default_factory=list)
    row_count: int = 0
    truncated: bool = False
    sql: str = ""
    entities: list[str] = Field(default_factory=list)
    unmet_parts: list[str] = Field(default_factory=list)
    grounding_record_ids: list[str] = Field(default_factory=list)
    result_note: str = ""
    repair_count: int = 0
    usage: TokenUsage = Field(default_factory=TokenUsage)

print("Schemas defined.")

## 2. Example: what `ModelDecision` contains

Imagine the user asks: **“Which products generated the most revenue last month?”**

After the agent has worked through its tools, it returns a judgment like this. These are illustrative values, not results from a live database.


In [ ]:
decision = ModelDecision(
    entities=["product 2297 (RAN-FIC-1296)", "product 4810 (RAN-FIC-2031)"],
    answerable=True,
    note="Revenue is merchandise total; shipping is excluded.",
    unmet_parts=[]
)

decision.model_dump()

Notice what is *not* in `ModelDecision`:

- No SQL statement.
- No result columns.
- No database rows.

It records the agent's judgment and caveats. It is not the final data payload.


## 3. Register structured output with `ToolStrategy`

In the real NLQ agent, the key configuration is:

```python
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from .schemas import ModelDecision

agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    state_schema=NLQState,
    response_format=ToolStrategy(schema=ModelDecision),
    middleware=[BudgetMiddleware(), GuardMiddleware(), GroundingMiddleware()],
    name="nlq",
)
```

`response_format=ToolStrategy(schema=ModelDecision)` tells LangChain to obtain a response that follows `ModelDecision`. With this strategy, structured output is handled through a tool-call strategy.

This is the actual pattern used in the project; `ProductRevenue` or other question-specific schemas are **not** needed here.


## 4. Where does the structured decision appear?

When the agent returns its final state, the parsed object is available under **`structured_response`**.

```python
result = await agent.ainvoke({
    "messages": [{"role": "user", "content": question}]
})

decision = result["structured_response"]  # ModelDecision instance
print(decision.answerable)
print(decision.entities)
print(decision.note)
```

In our streaming implementation, `core.py` watches update events and captures `update["structured_response"]` when present. That decision is then passed to `assemble()`.

**Do not confuse these three things:**
- `result` — the full agent state/output dictionary.
- `result["structured_response"]` — the parsed `ModelDecision`.
- `NLQResponse` — the final application response assembled afterwards.


## 5. Assemble the final response from trusted tool state

This simplified version follows the same principle as the project's `assemble.py`: the final rows and SQL come from the captured tool state, not from the model's structured judgment.


In [ ]:
def assemble_demo(
    decision: ModelDecision,
    captured: dict | None,
    executed_sql: str,
    grounding_ids: list[str] | None = None,
    repair_count: int = 0,
) -> NLQResponse:
    common = {
        "entities": decision.entities,
        "unmet_parts": decision.unmet_parts,
        "grounding_record_ids": grounding_ids or [],
        "repair_count": repair_count,
        "sql": executed_sql or "",
    }

    if not decision.answerable:
        return NLQResponse(
            result_shape="unanswerable",
            result_note=decision.note,
            **common,
        )

    if not captured:
        return NLQResponse(
            result_shape="not_executed",
            result_note=decision.note or "no query was executed",
            **common,
        )

    rows = captured.get("rows") or []
    note = decision.note
    if captured.get("truncated"):
        note = (note + " " if note else "") + (
            f"Result capped at {len(rows)} rows; more rows exist."
        )

    return NLQResponse(
        result_shape="table" if rows else "empty",
        columns=captured.get("columns") or [],
        rows=rows,
        row_count=len(rows),
        truncated=bool(captured.get("truncated")),
        result_note=note.strip(),
        **common,
    )

### 5A. A query ran and returned rows

The values below are **mock tool output** so we can run the example offline. In the real application, `execute_sql` captures the database output in agent state.


In [ ]:
captured_tool_result = {
    "columns": ["product_id", "sku", "revenue"],
    "rows": [
        [2297, "RAN-FIC-1296", 18450.75],
        [4810, "RAN-FIC-2031", 16220.00],
    ],
    "truncated": False,
}

final_response = assemble_demo(
    decision=decision,
    captured=captured_tool_result,
    executed_sql="SELECT product_id, sku, SUM(line_total) AS revenue FROM order_items GROUP BY product_id, sku ORDER BY revenue DESC LIMIT 100",
    grounding_ids=["schema-products-01", "metric-revenue-02"],
    repair_count=1,
)

final_response.model_dump()

### 5B. What if the model says the question is unanswerable?

The application can return an `unanswerable` response without pretending there are database rows.


In [ ]:
unanswerable_decision = ModelDecision(
    answerable=False,
    note="The schema does not contain customer lifetime satisfaction scores."
)

unanswerable_response = assemble_demo(
    decision=unanswerable_decision,
    captured=None,
    executed_sql="",
)

unanswerable_response.model_dump()

### 5C. What if no query ran?

If the agent does not execute a query and has not marked the question unanswerable, the application reports `not_executed`. It does not invent a table.


In [ ]:
not_executed_response = assemble_demo(
    decision=ModelDecision(note="No query result was captured."),
    captured=None,
    executed_sql="",
)

not_executed_response.model_dump()

## 6. The end-to-end flow

```text
User question
    |
    v
NLQ agent + tools
    |                           | model's judgment       \ execute_sql writes results to state
    v                         v
ModelDecision             captured rows + executed SQL
    |                         |
    +------------+------------+
                 v
             assemble()
                 |
                 v
           NLQResponse
                 |
                 v
       Supervisor / chart agent
```

### Why split the contract?

1. **Clear responsibilities:** the model supplies judgments; tools capture query results; Python assembles the response.
2. **More reliable data:** the final rows and SQL are sourced from tool state, not generated as free-form model fields.
3. **Predictable downstream interface:** the supervisor and chart agent receive consistent fields such as `columns`, `rows`, and `result_shape`.
4. **Useful failure states:** `empty`, `unanswerable`, and `not_executed` mean different things.


## 7. Important limitation

Structured output validates the **shape and types** of a response. It does **not** prove that a model's judgment is factually correct.

For example, Pydantic can check that `answerable` is a boolean and `entities` is a list of strings. It cannot independently verify that the SQL is correct or that a business interpretation is right. That is why this design keeps actual rows and executed SQL tied to the tool/state path and uses grounding, SQL validation, and execution controls separately.

## Quick recap

- `ModelDecision` = the model's structured judgment.
- `ToolStrategy(schema=ModelDecision)` = the LangChain agent configuration for structured output.
- `structured_response` = the parsed `ModelDecision` in the agent result/state.
- `assemble()` = combines the decision with captured tool state.
- `NLQResponse` = the final tabular contract passed to the supervisor.

**Next topic:** Model Wrapping.
